# Credit Scoring Scorecard — Revised Workflow
**Miftahul Jannat Jessica**

This notebook uses the same tested package as the command line. The original notebook and its saved 70/30 results are preserved in `original_credit_scoring.ipynb`. This workflow makes a **new** approximately 60/20/20 grouped split, fits preprocessing and WoE on training only, freezes cutoff values on validation and evaluates them on test. Its metrics must not be mixed with the historical 0.853 AUC.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    if (ROOT.parent / 'pyproject.toml').exists():
        ROOT = ROOT.parent
    else:
        ROOT = Path('/content/credit-scoring-scorecard')
        if not ROOT.exists():
            subprocess.run(['git','clone','https://github.com/Jessica-8-mifta/credit-scoring-scorecard.git',str(ROOT)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], cwd=ROOT, check=True)
sys.path.insert(0, str(ROOT / 'src'))
print('Project:', ROOT)


## Run on your original data
Place the source ZIP under `data/train.csv.zip` or change `DATA` below. The ZIP must contain one CSV with the named target and ten predictors described in `docs/DATA.md`. Extra index columns are ignored. No Kaggle credentials are embedded or required once you have the local data.

Set `RUN_REAL_DATA=True` after checking the path. Use a fresh run directory each time; runs are never overwritten. Bootstrap settings and forest size are fixed before inspecting test results. The 30% validation rejection quantile is an illustrative policy, not a profitability optimum.


In [ ]:
from credit_scorecard.pipeline import run_experiment

DATA = ROOT / 'data' / 'train.csv.zip'
OUTPUT = ROOT / 'runs' / 'baseline'
RUN_REAL_DATA = False

if RUN_REAL_DATA:
    run_experiment(DATA, OUTPUT, seed=42, trees=300, bootstrap=500, reject_share=0.30)
else:
    print('Real-data run disabled. Supply train.csv.zip and enable RUN_REAL_DATA.')


## Inspect outputs from this run
`metrics.json` contains computed AUC/Gini/KS, Brier score, log loss, a constant-prevalence baseline, paired bootstrap AUC intervals and policy outcomes. `bin_audit.csv` and `iv_diagnostics.csv` expose training support and stability diagnostics. High IV is a reason to inspect bins; it does not alone prove leakage. Calibration is assessed, not automatically corrected.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image, HTML

if (OUTPUT / 'metrics.json').exists():
    results = json.loads((OUTPUT / 'metrics.json').read_text())
    print(json.dumps(results, indent=2))
    display(pd.read_csv(OUTPUT / 'scorecard_points.csv').head(12))
    display(pd.read_csv(OUTPUT / 'iv_diagnostics.csv'))
    display(Image(filename=str(OUTPUT / 'dashboard.png')))
else:
    print('No new real-data results yet. The original saved outputs remain in the historical notebook.')


## Exact scorecard and hypothetical worked example
The score is 600 at good-to-bad odds of 50:1; doubling those odds adds 20 points. Exact bin-point sums are checked against model scores, while separately rounded points may differ by up to 3 points. Policy membership always uses the exact score. The worked example is a hypothetical record, not a disclosed borrower's information.


In [ ]:
if (OUTPUT / 'worked_example.json').exists():
    example = json.loads((OUTPUT / 'worked_example.json').read_text())
    print(json.dumps(example, indent=2))
    display(pd.read_csv(OUTPUT / 'cutoff_table.csv'))


## Optional synthetic execution demo
This exercises the real code on engineered data, using a smaller computation budget. It is **not evidence of consumer-credit predictive performance** and is never used to replace original results. Enable it separately if the original source data is unavailable.


In [ ]:
RUN_SYNTHETIC_DEMO = False

if RUN_SYNTHETIC_DEMO:
    subprocess.run([sys.executable,'-m','credit_scorecard','demo','--output',str(ROOT/'runs'/'synthetic_demo')], cwd=ROOT, check=True)


## Preserve and interpret the evidence
Back up `model.joblib`, its checksum, the exact installed versions, source checksum, split manifest, points/bin tables, calibration exports and metric/report files. Only load trusted serialized models. Do not publish raw borrower data.

AUC evaluates ranking; a probability-calibration plot evaluates agreement with observed rates. Historical accepted bad rates do not establish profitability, regulatory suitability, time stability or fairness. Once you inspect the final test, do not keep tuning against it and call it untouched. See `docs/METHODOLOGY.md` for the full evaluation protocol.
